In [1]:
import sys
from pathlib import Path
import pandas as pd

projectRoot = Path().resolve().parents[0]
print(f"path: {projectRoot}")
sys.path.append(str(projectRoot))

%load_ext autoreload
%autoreload 2

path: /workspace


In [ ]:
# Extract canonical facts from FCA HTML files
from src.parsing.arelle_parser import process_html_files

html_folder = projectRoot / "data" / "raw" / "fca"
output_path = (projectRoot / "data" / "processed" / 
               "canonical_facts" / "bronze.csv")

if output_path.exists():
    print(f"Bronze data already exists at: {output_path}")
    bronze_df = pd.read_csv(output_path)
else:
    bronze_df = process_html_files(html_folder, output_path, debug=False)

sample_bronze = bronze_df.sample(n=250, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_bronze.csv")
sample_bronze.to_csv(csv_path, index=False)
print(f"Bronze sample extract saved to: {csv_path}")

Bronze data saved to: /workspace/data/processed/canonical_facts/bronze.csv
Bronze sample extract saved to: /workspace/data/processed/debug/sample_bronze.csv


In [3]:
# Create silver ground truth from bronze data
from src.parsing.canonical_facts import create_silver_ground_truth

silver_df = create_silver_ground_truth(bronze_df, 
                        str(projectRoot / "data" / "processed" /
                            "canonical_facts" / "silver.csv"))

sample_silver = silver_df.sample(n=300, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_silver.csv")
sample_silver.to_csv(csv_path, index=False)
print(f"Silver sample extract saved to: {csv_path}")

Created cleaned ground truth with 5174 rows.
Silver data saved to: /workspace/data/processed/canonical_facts/silver.csv
Silver sample extract saved to: /workspace/data/processed/debug/sample_silver.csv


In [4]:
# Create gold ground truth from silver data
from src.parsing.canonical_facts import create_gold_ground_truth

gold_df = create_gold_ground_truth(silver_df, 
                         str(projectRoot / "data" / "processed" /
                              "canonical_facts" / "gold.csv"))

sample_gold = gold_df.sample(n=300, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_gold.csv")
sample_gold.to_csv(csv_path, index=False)
print(f"Gold sample extract saved to: {csv_path}")

Gold data saved to: /workspace/data/processed/canonical_facts/gold.csv
Gold sample extract saved to: /workspace/data/processed/debug/sample_gold.csv


In [5]:
sample_gold_tiny = gold_df[~gold_df['segment'].isin(['Narrative_Disclosure', 
                        'Other_Financial_Metric'])].sample(n=10, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_gold_tiny.csv")
sample_gold_tiny.to_csv(csv_path, index=False)
print(f"Tiny Gold sample extract saved to: {csv_path}")

Tiny Gold sample extract saved to: /workspace/data/processed/debug/sample_gold_tiny.csv


In [ ]:
from src.qa_generation.llm_qa_generator import generate_qa_openai
input_csv_path = (projectRoot / "data" / "processed" /
                  "debug" / "sample_gold_tiny.csv")
output_csv_path = (projectRoot / "data" / "processed" /
                   "debug" / "sample_qa_pairs_tiny.csv")
qa_pairs_df = generate_qa_openai(input_csv_path, output_csv_path)